In [2]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta
import os

fake = Faker('en_NZ')
random.seed(42)
np.random.seed(42)

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [3]:
branches = pd.DataFrame([
    {'branch_id': 1,  'branch_name': 'BNT Auckland Central',      'brand': 'BNT',          'region': 'Auckland',     'segment': 'Trade'},
    {'branch_id': 2,  'branch_name': 'BNT Christchurch',          'brand': 'BNT',          'region': 'Christchurch', 'segment': 'Trade'},
    {'branch_id': 3,  'branch_name': 'BNT Wellington',            'brand': 'BNT',          'region': 'Wellington',   'segment': 'Trade'},
    {'branch_id': 4,  'branch_name': 'Autolign Auckland',         'brand': 'Autolign',     'region': 'Auckland',     'segment': 'Specialist Wholesale'},
    {'branch_id': 5,  'branch_name': 'Autolign Wellington',       'brand': 'Autolign',     'region': 'Wellington',   'segment': 'Specialist Wholesale'},
    {'branch_id': 6,  'branch_name': 'HCB Technologies Auckland', 'brand': 'HCB',          'region': 'Auckland',     'segment': 'Specialist Wholesale'},
    {'branch_id': 7,  'branch_name': 'Battery Town Auckland',     'brand': 'Battery Town', 'region': 'Auckland',     'segment': 'Service'},
    {'branch_id': 8,  'branch_name': 'Battery Town Hamilton',     'brand': 'Battery Town', 'region': 'Hamilton',     'segment': 'Service'},
    {'branch_id': 9,  'branch_name': 'Diesel Distributors AKL',  'brand': 'Diesel Dist',  'region': 'Auckland',     'segment': 'Specialist Wholesale'},
    {'branch_id': 10, 'branch_name': 'BNT Hamilton',              'brand': 'BNT',          'region': 'Hamilton',     'segment': 'Trade'},
])

print(f"Branches created: {len(branches)}")
print(branches)

Branches created: 10
   branch_id                branch_name         brand        region  \
0          1       BNT Auckland Central           BNT      Auckland   
1          2           BNT Christchurch           BNT  Christchurch   
2          3             BNT Wellington           BNT    Wellington   
3          4          Autolign Auckland      Autolign      Auckland   
4          5        Autolign Wellington      Autolign    Wellington   
5          6  HCB Technologies Auckland           HCB      Auckland   
6          7      Battery Town Auckland  Battery Town      Auckland   
7          8      Battery Town Hamilton  Battery Town      Hamilton   
8          9    Diesel Distributors AKL   Diesel Dist      Auckland   
9         10               BNT Hamilton           BNT      Hamilton   

                segment  
0                 Trade  
1                 Trade  
2                 Trade  
3  Specialist Wholesale  
4  Specialist Wholesale  
5  Specialist Wholesale  
6              

In [7]:
categories = {
    'Brakes':     {'brands': ['Bendix', 'DBA', 'RDA'],       'min_cost': 15,  'max_cost': 180},
    'Filters':    {'brands': ['Ryco', 'Baldwin', 'Mann'],     'min_cost': 8,   'max_cost': 45},
    'Suspension': {'brands': ['Monroe', 'KYB', 'Pedders'],    'min_cost': 40,  'max_cost': 320},
    'Batteries':  {'brands': ['Century', 'Optima', 'Bosch'],  'min_cost': 80,  'max_cost': 280},
    'Electrical': {'brands': ['Bosch', 'Denso', 'NGK'],       'min_cost': 12,  'max_cost': 150},
    'Engine':     {'brands': ['Castrol', 'Penrite', 'Nulon'], 'min_cost': 20,  'max_cost': 95},
}

products = []
product_id = 1

for category, info in categories.items():
    for i in range(20):
        cost = round(random.uniform(info['min_cost'], info['max_cost']),2)
        margin = random.uniform(1.3,1.8)
        products.append({
            'product_id' : product_id,
            'part_number': f"BNT-{category[:3].upper()}-{1000 + product_id}",
            'category': category,
            'brand': random.choice(info['brands']),
            'unit_cost': cost,
            'rrp': round(cost*margin,2),
        })
        product_id += 1

products_df = pd.DataFrame(products)

print(f"Products created: {len(products_df)}")
print("\nAverage RRP by category:")
print(products_df.groupby('category')['rrp'].mean().round(2))
print("\nSample rows:")
print(products_df.head(3))

Products created: 120

Average RRP by category:
category
Batteries     289.41
Brakes        117.90
Electrical    120.93
Engine         84.83
Filters        41.96
Suspension    282.58
Name: rrp, dtype: float64

Sample rows:
   product_id   part_number category   brand  unit_cost    rrp
0           1  BNT-BRA-1001   Brakes     RDA      60.38  85.23
1           2  BNT-BRA-1002   Brakes     RDA      31.91  53.30
2           3  BNT-BRA-1003   Brakes  Bendix      29.34  44.33


In [8]:
customer_types = ['Independent Workshop', 'Chain Mechanic', 'Retail', 'Fleet Manager']
customer_weights = [0.45, 0.25, 0.20, 0.10]

customers = []

for i in range(200):
    join_date = fake.date_between(start_date='-2y', end_date='-3m')
    customers.append({
        'customer_id': i+1,
        'customer_name': fake.company(),
        'customer_type': random.choices(customer_types, weights = customer_weights)[0],
        'region': random.choices(
            ['Auckland', 'Wellington', 'Christchurch', 'Hamilton'],
            weights=[0.45, 0.25, 0.20, 0.10]
        )[0],
        'join_date': join_date
    })

customers_df = pd.DataFrame(customers)

print(f"Customers created: {len(customers_df)}")
print("\nCustomer type breakdown:")
print(customers_df['customer_type'].value_counts())
print("\nRegion breakdown:")
print(customers_df['region'].value_counts())



Customers created: 200

Customer type breakdown:
customer_type
Independent Workshop    84
Chain Mechanic          52
Retail                  48
Fleet Manager           16
Name: count, dtype: int64

Region breakdown:
region
Auckland        94
Wellington      52
Christchurch    36
Hamilton        18
Name: count, dtype: int64


In [9]:
orders = []
order_id = 1

start_date = datetime(2024,1,1)
end_date = datetime(2025,12,31)

for _ in range(5000):
    order_date = fake.date_time_between(start_date=start_date,end_date=end_date)
    month = order_date.month

    # Winter seasonality (June-Aug): batteries spike, filters drop
    if month in [6,7,8]:
        category_weights = [0.15, 0.10, 0.15, 0.30, 0.15, 0.15]
    # Summer (Dec-Feb): filters and engine oil spike
    elif month in [12,1,2]:
        category_weights = [0.15, 0.30, 0.15, 0.10, 0.15, 0.15]
    # Rest of year: balanced
    else:
        category_weights = [0.20, 0.20, 0.18, 0.12, 0.18, 0.12]

    category_map = dict(zip(
        ['Brakes', 'Filters', 'Suspension', 'Batteries', 'Electrical', 'Engine'],
        category_weights
    ))

    product = products_df.sample(
        1,
        weights=products_df['category'].map(category_map)
    ).iloc[0]

    branch = branches.sample(1).iloc[0]
    customer = customers_df.sample(1).iloc[0]
    quantity = random.choices([1,2,3,4,5], weights=[0.40, 0.30, 0.15, 0.10, 0.05])[0]

    orders.append({
        'order_id':    order_id,
        'order_date':  order_date.date(),
        'branch_id':   int(branch['branch_id']),
        'customer_id': int(customer['customer_id']),
        'product_id':  int(product['product_id']),
        'quantity':    quantity,
        'unit_price':  float(product['rrp']),
        'revenue':     round(float(product['rrp']) * quantity, 2),
    })

    order_id += 1

orders_df = pd.DataFrame(orders)
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])

print(f"Orders created: {len(orders_df)}")
print(f"Total revenue: ${orders_df['revenue'].sum():,.2f}")
print(f"Date range: {orders_df['order_date'].min().date()} to {orders_df['order_date'].max().date()}")
print(f"\nRevenue by category:")

category_revenue = orders_df.merge(products_df[['product_id','category']], on='product_id')
print(category_revenue.groupby('category')['revenue'].sum().sort_values(ascending=False).apply(lambda x: f"${x:,.2f}"))

    

Orders created: 5000
Total revenue: $1,627,805.32
Date range: 2024-01-01 to 2025-12-30

Revenue by category:
category
Batteries     $498,036.45
Suspension    $493,504.73
Brakes        $217,452.74
Electrical    $205,933.37
Engine        $123,890.42
Filters        $88,987.61
Name: revenue, dtype: str


In [11]:
os.makedirs('data', exist_ok=True)

branches.to_csv('data/branches.csv', index=False)
products_df.to_csv('data/products.csv', index=False)
customers_df.to_csv('data/customers.csv', index=False)
orders_df.to_csv('data/orders.csv', index=False)

print("All files saved to /data folder:")
print(f"  branches.csv   — {len(branches)} rows")
print(f"  products.csv   — {len(products_df)} rows")
print(f"  customers.csv  — {len(customers_df)} rows")
print(f"  orders.csv     — {len(orders_df)} rows")
print(f"\nTotal dataset size: {len(branches) + len(products_df) + len(customers_df) + len(orders_df)} rows")

All files saved to /data folder:
  branches.csv   — 10 rows
  products.csv   — 120 rows
  customers.csv  — 200 rows
  orders.csv     — 5000 rows

Total dataset size: 5330 rows
